<a href="https://colab.research.google.com/github/dhaya0/stats_4/blob/main/CSSL_07_Regularization_and_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import zipfile

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV

!pip install ISLP
from ISLP import load_data

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.9 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=993e334f0a73b5bfcbba7eb8bb3059adea4954ddf9661cae8f45e7d7fe6b7799
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


# Utility functions

In [2]:
# Compute RMSE
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Load data

In [3]:
df = load_data('Hitters')

Now lets take a look at the data

In [14]:
print("Shape:", df.shape)
df.head()
type(df)

Shape: (263, 20)


pandas.core.frame.DataFrame

# Preprocessing
We need to find if there are missing values in the dataset and figure out a way to deal with them.

In [8]:
rows_with_na = df[df.isna().any(axis=1)]
print("\nRows containing NAs:")
len(rows_with_na)


Rows containing NAs:


59

Let's drop the rows with NaN values and split the dataset into train and test.

In [9]:
df = df.dropna().reset_index(drop=True)
target = "Salary"
X = df.drop(columns=[target])
y = df[target]
print(X.shape, y.shape)

# First split: train+val vs test (80% / 20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10
)

# Second split: train vs validation (75% / 25% of remaining 80%)
# This gives: 60% train, 20% val, 20% test overall
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=20
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

(263, 19) (263,)
Train size: 157
Validation size: 53
Test size: 53


Now we need to transform the categorical values into numerical values and also scale all the features so that they roughly lie within the same range. If some features are represented using large numbers while others are represented in small numbers, it can skew the error calculcations, similar to what happened when we did polynomial regression.

In [16]:
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols), # Scales columns containing numerical data using (x-mu)/sigma so that the values have mean 0 and standard deviation 1
        ("cat", OneHotEncoder(drop="first"), categorical_cols) # Encodes categorical values using one hot encoding
    ]
)

print(numerical_cols)
print(categorical_cols)


['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks', 'PutOuts', 'Assists', 'Errors']
['League', 'Division', 'NewLeague']


In [22]:
# If you want to understand how the OneHotEncoder works, uncomment the below lines of code and try it out
X_1hot = [["East"], ["West"], ["Central"]]

enc = OneHotEncoder(drop="first", sparse_output=False) # Note- The encoder sorts the list of unique categories before encoding
encoded = enc.fit_transform(X_1hot)

print(encoded)
# If East - West - Central
# To represent West = [0 1 0]
# Adds collinearity
# Cannot use 0,1,2 because it is like ordering the directions

[[1. 0.]
 [0. 1.]
 [0. 0.]]


# Linear regression using OLS

In [26]:
# OLS Model
ols_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

# 2) Fit on training data
ols_model.fit(X_train_val, y_train_val)

# 3) Predictions
y_train_pred = ols_model.predict(X_train_val)
y_test_pred  = ols_model.predict(X_test)


ols_test_rmse = rmse(y_test, y_test_pred)

print("OLS Performance:")
print("Train RMSE:", rmse(y_train, y_train_pred))
print("Test RMSE:", rmse(y_test, y_test_pred))

OLS Performance:


ValueError: Found input variables with inconsistent numbers of samples: [157, 210]

In [32]:
# Define lambdas grid
lambdas = np.logspace(-2, 3, 100)
print(lambdas)

# 5-fold CV
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

ridge_val_rmse = np.full((num_folds, len(lambdas)), np.inf)


for l_ind, l in enumerate(lambdas):
  fold = 0

  for train_idx, val_idx in kf.split(X_train_val):

    X_train_fold = X_train_val.iloc[train_idx]
    y_train_fold = y_train_val.iloc[train_idx]
    X_val_fold   = X_train_val.iloc[val_idx]
    y_val_fold   = y_train_val.iloc[val_idx]

    # Pipeline: preprocessing + Ridge
    ridge_model = Pipeline([
            ("preprocess", preprocess),
            ("ridge", Ridge(alpha=l, max_iter=10000))
        ])

    ridge_model.fit(X_train_fold, y_train_fold)
    ridge_preds = ridge_model.predict(X_val_fold)

    # Pipeline: preprocessing + Lasso
    lasso_model = Pipeline([
            ("preprocess", preprocess),
            ("lasso", Lasso(alpha=l, max_iter=10000))
    ])

    lasso_model.fit(X_train_fold, y_train_fold)
    lasso_preds = lasso_model.predict(X_val_fold)

    lasso_val_rmse = rmse(y_val_fold, lasso_preds)

    fold += 1
    lasso_mean_rmse = np.mean(lasso_val_rmse, axis=0)#?

    ridge_val_rmse[fold, l_ind] = rmse(y_val_fold, ridge_preds)

    fold += 1

    ridge_mean_rmse = np.mean(ridge_val_rmse, axis=0)

print(ridge_mean_rmse)

# Find best lambda for Ridge
ridge_best_lambda = lambdas[np.argmin(ridge_mean_rmse)]
print("Best lambda for ridge:", ridge_best_lambda)
print("Best Validation RMSE for ridge:", min(ridge_mean_rmse))

# Find best lambda for Lasso
lasso_best_lambda = lambdas[np.argmin(lasso_mean_rmse)]
print("Best lambda for lasso:", lasso_best_lambda)
print("Best Validation RMSE for lasso:", min(lasso_mean_rmse))

# Plot the lambdas vs mean_rmse for ridge and lasso
plt.figure()
plt.plot(lambdas, ridge_mean_rmse, c="b", label="ridge")

plt.xscale('log')
plt.legend()

[1.00000000e-02 1.12332403e-02 1.26185688e-02 1.41747416e-02
 1.59228279e-02 1.78864953e-02 2.00923300e-02 2.25701972e-02
 2.53536449e-02 2.84803587e-02 3.19926714e-02 3.59381366e-02
 4.03701726e-02 4.53487851e-02 5.09413801e-02 5.72236766e-02
 6.42807312e-02 7.22080902e-02 8.11130831e-02 9.11162756e-02
 1.02353102e-01 1.14975700e-01 1.29154967e-01 1.45082878e-01
 1.62975083e-01 1.83073828e-01 2.05651231e-01 2.31012970e-01
 2.59502421e-01 2.91505306e-01 3.27454916e-01 3.67837977e-01
 4.13201240e-01 4.64158883e-01 5.21400829e-01 5.85702082e-01
 6.57933225e-01 7.39072203e-01 8.30217568e-01 9.32603347e-01
 1.04761575e+00 1.17681195e+00 1.32194115e+00 1.48496826e+00
 1.66810054e+00 1.87381742e+00 2.10490414e+00 2.36448941e+00
 2.65608778e+00 2.98364724e+00 3.35160265e+00 3.76493581e+00
 4.22924287e+00 4.75081016e+00 5.33669923e+00 5.99484250e+00
 6.73415066e+00 7.56463328e+00 8.49753436e+00 9.54548457e+00
 1.07226722e+01 1.20450354e+01 1.35304777e+01 1.51991108e+01
 1.70735265e+01 1.917910

AxisError: axis 0 is out of bounds for array of dimension 0

# Fit the final model
Now that we have the best lambda values, we can try to fit the whole train+val dataset using these values.

In [29]:
ridge_final_model = Pipeline([
    ("preprocess", preprocess),
    ("ridge", Ridge(alpha=ridge_best_lambda))
])

ridge_final_model.fit(X_train_val, y_train_val)

ridge_test_preds = ridge_final_model.predict(X_test)



print("Test RMSE for OLS:", ols_test_rmse)
print("Test RMSE for Ridge Regression with cross validation:", rmse(y_test, ridge_test_preds))


Test RMSE for OLS: 309.4980873341895
Test RMSE for Ridge Regression with cross validation: 296.1859621013697


Get the coefficients of all parameters; get it for the original ols model, then ridge, then lasso


use help fn for everything